
$\qquad$ $\qquad$$\qquad$  **TDA206/DIT206 Discrete Optimization: Home Assignment 2 -- Integer LP and Relaxation** <br />
$\qquad$ $\qquad$$\qquad$                   **Grader: Marc Constantin** <br />
$\qquad$ $\qquad$$\qquad$                   **Due Date: 17th Feb** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Name, Personal No., Email** <br />


---


General guidelines:
*   All solutions to theoretical and pratical problems must be submitted in this ipynb notebook, and equations wherever required, should be formatted using LaTeX math-mode.
*   All discussion regarding practical problems, along with solutions and plots should be specified in this notebook. All plots/results should be visible such that the notebook do not have to be run. But the code in the notebook should reproduce the plots/results if we choose to do so.
*   Your name, personal number and email address should be specified above.
*   All tables and other additional information should be included in this notebook.
*   Before submitting, make sure that your code can run on another computer. That all plots can show on another computer including all your writing. It is good to check if your code can run here: https://colab.research.google.com.


# Question 1.
(5 points) 
There are 4 space colonies, each of which  requires a certain number of plasma conduits. There are 3 starbases in the vicinity. Each of them has total number of conduits they can spare and supply to the colonies. For each pair of starbase and colony, there is an associated cost for sending a cargo ship  (each of which carries one plasma conduit), as shown in the table below:


\begin{array}{l|c|c|c|c|c} 
      & Triacus & New Berlin  & Strnad  & Vega  & supply\\ \hline
 Farpoint &   6 &  9 & 10 & 8 & 35\\
 Yorktown &  9 & 5 & 16 & 14 & 40\\
 Earhart & 12 &  7 & 13 & 9 & 50\\ \hline
    demand & 20 &30&30&45& \left(\sum=125\right) \\ 
\end{array}

Your goal is to supply the colonies the plasma conduits they need, at minimum cost.


Formulate a LP to solve the problem, explain why the solution is integral with a proof.

# 1 
We want to fill the demand of plasma tubes at the lowest cost. This can be expressed as:
$$
min \sum_{i = 1}^{3} \sum_{j = 1}^{4} c_{ij}x_{ij}
$$
Where $c_{ij}$ is the cost to deliver one plasma tube from starbase $i$ to space colony $j$.
This is subject to the constraints:

$$
\begin{align*} 
 \sum^{4}_{j = 1} x_{1j}  & = 35 \\
 \sum^{4}_{j = 1} x_{2j}  & = 40 \\
 \sum^{4}_{j = 1} x_{3j}  & = 50  \\
\sum^{3}_{i=1}x_{i1}  & = 20 \\
\sum^{3}_{i=1}x_{i 2}  & = 30 \\
\sum^{3}_{i=1} x_{i3}  & = 30 \\
\sum^{3}_{i = 1} x_{i 4}  & =  40 \\
x \geq 0
\end{align*}
$$


In [8]:
import cvxpy as cp
import numpy as np

x = cp.Variable((3,4), nonneg = True)
c = np.array([[6, 9, 10, 8],
              [9, 5, 16, 14],
              [12, 7, 13, 9]])
objective = cp.Minimize(cp.sum(cp.multiply(c, x)))

constraint = [
    cp.sum(x, axis=1) == [35, 40, 50],   # supplies
    cp.sum(x, axis=0) == [20, 30, 30, 45]  # demands
             ]

problem = cp.Problem(objective, constraint)
problem.solve()

print("Status:", problem.status)
print("Optimal value:", problem.value)
print(np.round(x.value))





Status: optimal
Optimal value: 1020.0000022723721
[[10.  0. 25.  0.]
 [10. 30.  0.  0.]
 [ 0.  0.  5. 45.]]


# Question 2.

Recall the Minimum Weight Vertex Cover (VC) Problem: Given an undirected graph $G=(V, E)$, with node set $V$ and edge set $E$, where each node has a positive weight $w(v)$ associated with it (see the attached figure), the goal is to select a subset $V'\subseteq V$ of nodes such that every edge has at least one node incident to it, and the total selected node weight $\sum_{v\in V'} w(v)$ is minimized. 

* (4 points) Formulate the ILP for the VC problem for the attached graph, and solve it using **CVXPY** integer solver, for instance, `myVar = cp.Variable(<dim>, integer=True)`.
* (2 points) Pass to the LP relaxation and solve it using **CVXPY** and comment on the relation between the two solutions.
* (2 points) Apply the rounding rule discussed in class to the optimal LP solution to obtain a solution to the ILP and compare it to the optimal ILP solution.

<!-- ![title](vertex_cover_example.png) -->


$$
\begin{align*}
\text{Minimize: } & \sum_{v \in V}w_{x}x_{v}  \\
\text{Subject to: } & x_{u} +x_{v} \geq 1 \quad \text{for every edge} \left\{ u,v \right\} \in E\\
 & x_{v} \in \left\{ 0,1 \right\} \quad \text{for all } v\in V
\end{align*}
$$

In [18]:
x = cp.Variable(6, integer=True)
edges = [
    (0,1), (1,2), (1,4), (2,3), (2,4), (3,4), (4,5)
        ] 
weights = np.array([1,3,4,2,2,4])

objective = cp.Minimize(weights @ x)                  
constraint = [
    x >= 0,
    x <= 1
]
for u,v in edges:
    constraint.append(x[u] + x[v] >= 1)
    
problem = cp.Problem(objective, constraint)
problem.solve()

print("Status:", problem.status)
print("Optimal value:", problem.value)
print("x value: ", x.value)


Status: optimal
Optimal value: 7.0
x value:  [-0.  1. -0.  1.  1. -0.]


In [23]:
###
# LP relaxation
###
x = cp.Variable(6)
edges = [
    (0,1), (1,2), (1,4), (2,3), (2,4), (3,4), (4,5)
        ] 
weights = np.array([1,3,4,2,2,4])

objective = cp.Minimize(weights @ x)                  
constraint = [
    x >= 0,
    x <= 1
]
for u,v in edges:
    constraint.append(x[u] + x[v] >= 1)
    
problem = cp.Problem(objective, constraint)
problem.solve()

print("Status:", problem.status)
print("Optimal value:", problem.value)
print("x value: ", x.value)

# Applying the rounding rule
approx_x = [1 if x >= 1/2 else 0 for x  in x.value]
print("Approx x: ", approx_x)

Status: optimal
Optimal value: 6.99999999983728
x value:  [5.00595661e-01 4.99404339e-01 5.00595661e-01 4.99404339e-01
 1.00000000e+00 9.83664839e-11]
Approx x:  [1, 0, 1, 0, 1, 0]


# Question 3.

Consider a number of interpreters (Olof, Petra, Qamar,
  Rachel, Soren and Tao), as well as a set of languages (Arab,
  Bengali, Cantonese, Dutch, English, French and German). Each
  interpreter speaks a number of different languages (abbreviated by
  first letter), and has a certain per-diem integer cost:

\begin{array}{lll}
Interpreter & Languages & Cost\\
O & ABD & 3\\
P & C & 1\\
Q & CDG & 1\\
R & B & 2\\
S & G & 4\\
T & EF & 1\\
\end{array}

* (2 points) A *hypergraph* is a structure $H = (V,E)$ where $V$ is a set of vertices and $E$ is a collection of subsets of $V$. The special case when all subsets $e \in E$ have size exactly $2$ corresponds to the familiar case of a graph. A vertex cover in such a hypergraph is a subset $U \subseteq V$ such that $e \cap U \not = \emptyset$ for each $e \in E$ (note that this reduces to the usual vertex cover in graphs). Show that the problem of finding interpreters can be formulated as a vertex cover problem in a sutable hypergraph.
* (4 points) Develop a ILP formulation to finding the vertex cover of minimum cost in a hypergraph. The hypergraph can be represented as a $|V| \times |E|$ binary matrix $A$ where $A[i,j] = 1$ iff vertex $i$ is in edge $j$ and 0 otherwise. The costs for vertices are in an array $\texttt{c}$ where the cost of picking vertex $i$ is $c[i]$. Use the ILP formulation for the VC problem to hire the cheapest set of interpreters such that all languages are covered. Input the data above manuallly and solve it using **CVXPY**'s integer solver.
* (2 points) Pass to the LP relaxation and solve it using **CVXPY**.
* (2 points) Explain why the two solutions above are same (different).


# Question 4. 
Consider the ILP and its LP relaxation corresponding to the VC problem for the graph $G$ given in the data file. This is a ***random graph*** $G(n,p)$ with $n=100$ vertices generated as follows: for each pair of vertices **independently**, we add an edge with probability $p=0.1$ (so the graph has about 1000 edges).

* **a**. (2 points) Find the optimal solution using **CVXPY**'s integer solver.
* **b**. (2 points) Solve the LP relaxation using **CVXPY** and apply the rounding rule discussed in class to obtain a vertex cover. Compare it to the optimal solution in part (a).
* **c**. (6 points) Consider the following rounding rule: we build up the vertex cover incrementally starting with $S:= \emptyset$. Now consider the edges in $G$ in any order. If an edge $(u,v)$ is already covered by a vertex in $S$, do nothing. Otherwise add to $S$ the vertex $u$ if $x^*(u) \geq x^*(v)$, or $v$ otherwise (where ${\bf x}^*$ is the LP optimum solution computed in part (b).  Comment why this also results in a vertex cover and has cost no more than that corresponding to the rounding rule in part (b). Compare the cost of the solution produced by this rule to the optimal solution.

In [26]:
import os
import cvxpy as cp
import numpy as np

edges = []
with open("graph.txt") as file:
    content = file.read()
    content = content.split("\n")
    for i, row in enumerate(content):
        for j, element in enumerate(row.replace(" ", "")):
            #print(ele)
            if element == "1":
               edges.append((i,j))

x = cp.Variable(100, boolean=True)
weights = np.array([1 for i in range(100)])

objective = cp.Minimize(weights @ x)

constraint = [
]
for u,v in edges:
    constraint.append(x[u] + x[v] >= 1)
    
problem = cp.Problem(objective, constraint)
problem.solve()

print("Status:", problem.status)
print("Optimal value:", problem.value)
print("x value: ", np.round(x.value, 2))

Status: optimal
Optimal value: 70.0
x value:  [ 1.  1.  1.  0.  1.  1.  1.  1. -0. -0.  1. -0.  1.  1. -0. -0.  1.  1.
 -0.  1.  1. -0.  1. -0.  1.  1. -0.  1.  1.  1.  1.  1. -0.  1.  1.  1.
  1.  1.  1.  1.  1.  1.  1. -0.  1.  1.  1.  1.  1.  1. -0. -0.  1.  1.
  1.  1.  1. -0.  1.  1. -0. -0.  1.  1. -0.  1. -0.  1.  1.  0.  1. -0.
  1.  1. -0.  1.  0.  1.  1.  1.  1.  1.  1. -0.  1.  1.  0.  1. -0.  1.
  1. -0.  1.  1.  0.  1.  1.  1. -0.  0.]


In [39]:
import os
import cvxpy as cp
import numpy as np

edges = []
with open("graph.txt") as file:
    content = file.read()
    content = content.split("\n")
    for i, row in enumerate(content):
        for j, element in enumerate(row.replace(" ", "")):
            #print(ele)
            if element == "1":
               edges.append((i,j))

x = cp.Variable(100)
weights = np.array([1 for i in range(100)])

objective = cp.Minimize(weights @ x)

constraint = [
    x >= 0,
    x <= 1
]
for u,v in edges:
    constraint.append(x[u] + x[v] >= 1)
    
problem = cp.Problem(objective, constraint)
problem.solve()

print("Status:", problem.status)
print("Optimal value:", problem.value)
#print("x value: ", x.value)
for i, val in enumerate(x.value):
    print(f"Node {i}: {val}")

# Applying the rounding rule
approx_x = [1 if x >= 1/2 else 0 for x  in x.value]
#print("Approx x: ", approx_x)
for i, val in enumerate(approx_x):
    print(f"Node {i}: {val}")
print("Rounding cost: ", np.sum(approx_x))

Status: optimal
Optimal value: 50.000000001021206
Node 0: 0.5000000000829566
Node 1: 0.5000000000034441
Node 2: 0.500000000018714
Node 3: 0.4999999999827798
Node 4: 0.5000000000123181
Node 5: 0.5000000000263178
Node 6: 0.5000000000262141
Node 7: 0.5000000000145978
Node 8: 0.4999999999967491
Node 9: 0.500000000000621
Node 10: 0.49999999996569106
Node 11: 0.49999999999090233
Node 12: 0.5000000000663047
Node 13: 0.4999999999963494
Node 14: 0.4999999999775187
Node 15: 0.5000000000182547
Node 16: 0.500000000010855
Node 17: 0.49999999999153416
Node 18: 0.49999999993427224
Node 19: 0.5000000000070478
Node 20: 0.5000000000615435
Node 21: 0.49999999997348027
Node 22: 0.5000000000128518
Node 23: 0.49999999999622313
Node 24: 0.5000000000025093
Node 25: 0.4999999999871437
Node 26: 0.5000000000183114
Node 27: 0.5000000000268848
Node 28: 0.5000000000086784
Node 29: 0.5000000000209479
Node 30: 0.5000000000753704
Node 31: 0.49999999997417993
Node 32: 0.49999999997642125
Node 33: 0.5000000000102744
Nod